# 📋 Odpowiedzi: RAG warsztat

*Wygenerowano automatycznie — nie commitować!*

---
### 1. Ćwiczenie 1: Porównaj substring vs context stuffing

Uruchom OBA podejścia na własnym pytaniu i porównaj wyniki.
W którym przypadku odpowiedź jest lepsza?

In [ ]:
# === Ćwiczenie 1 — rozwiązanie ===

MOJE_PYTANIE = "kto miał związek z wojskiem?"

wynik_substring = search_presidents(MOJE_PYTANIE)
wynik_stuffing = context_stuffing_search(MOJE_PYTANIE) if client else "(LLM niedostępny)"

print(f'Pytanie: "{MOJE_PYTANIE}"\n')
print(f"Substring search:\n  {wynik_substring[:200]}\n")
print(f"Context stuffing:\n  {wynik_stuffing[:200]}")

---
### 2. Ćwiczenie 2: Eksperymentuj z rozmiarem fragmentów

Spróbuj zmienić parametry `chunk_size` i `overlap` w komórce powyżej.

> **`overlap`** — ile znaków "zachodzi" na siebie między sąsiednimi fragmentami.
> Np. `overlap=50` przy `chunk_size=300` oznacza, że ostatnie 50 znaków fragmentu N
> pojawia się też na początku fragmentu N+1. Dzięki temu zdanie "przecięte" na granicy
> chunka nie ginie — RAG może je znaleźć z obu stron.

**Proszę uzupełnić poniższą komórkę** — uruchom chunking z innymi parametrami i porównaj wyniki:

In [ ]:
# === Ćwiczenie 2 — rozwiązanie ===

chunks_male = chunk_text(full_text, chunk_size=200, overlap=30)
chunks_duze = chunk_text(full_text, chunk_size=1000, overlap=100)

print(f"Domyślne (500 znaków): {len(chunks)} fragmentów")
print(f"Małe (200 znaków):     {len(chunks_male)} fragmentów")
print(f"Duże (1000 znaków):    {len(chunks_duze)} fragmentów")

print(f"\nPrzykładowy mały fragment:")
print(f"  '{chunks_male[0][:100]}...'")
print(f"\nPrzykładowy duży fragment:")
print(f"  '{chunks_duze[0][:100]}...'")

---
### 3. Ćwiczenie 3: Cosine similarity "ręcznie"

Zanim użyjemy gotowej funkcji wyszukiwania, policzmy cosine similarity sami.

Przypomnijmy wzór:

$$\text{cosine\_similarity}(A, B) = \frac{A \cdot B}{\|A\| \cdot \|B\|}$$

Gdzie $A \cdot B$ to iloczyn skalarny, a $\|A\|$ to norma (długość) wektora.

In [ ]:
# === Ćwiczenie 3 — rozwiązanie ===

pytanie = "Kto otrzymał Pokojową Nagrodę Nobla?"
emb_pytania = embed_model.encode([pytanie])[0]
emb_fragment_a = chunk_embeddings[23]   # Wałęsa — Nobel
emb_fragment_b = chunk_embeddings[5]    # Jaruzelski — wojsko

similarity_a = np.dot(emb_pytania, emb_fragment_a) / (np.linalg.norm(emb_pytania) * np.linalg.norm(emb_fragment_a))
similarity_b = np.dot(emb_pytania, emb_fragment_b) / (np.linalg.norm(emb_pytania) * np.linalg.norm(emb_fragment_b))

print(f"Pytanie: '{pytanie}'\n")
print(f"Fragment A (Wałęsa — Nobel):")
print(f"  '{chunks[23][:100]}...'")
print(f"  Cosine similarity: {similarity_a:.4f}\n")
print(f"Fragment B (Jaruzelski — wojsko):")
print(f"  '{chunks[5][:100]}...'")
print(f"  Cosine similarity: {similarity_b:.4f}\n")
print(f"Który fragment jest bliższy pytaniu? {'Fragment A ✓' if similarity_a > similarity_b else 'Fragment B'}")
print(f"\nRóżnica: {abs(similarity_a - similarity_b):.4f} — embeddingi wyraźnie 'wiedzą' który tekst pasuje!")